# Training and Evaluating Learning-Based Models

This notebook demonstrates the complete workflow for:
1. Training a learning-based shape matching model **using presets** (RECOMMENDED)
2. Evaluating the trained model using the experiment framework
3. Comparing trained models with classical matchers

**Key insight**: Just like `MatcherPresets` for classical methods, we have `ModelPresets` and `TrainingPresets` for learning-based methods, making training as simple as experiments!

**Topics covered:**
1. Quick training with `quick_train()` (one-liner setup)
2. Step-by-step training with presets
3. Loading and evaluating trained models
4. Comparing trained models with classical matchers
5. Manual training setup (advanced)

## Setup

In [ ]:
import os

os.environ["GEOMSTATS_BACKEND"] = "pytorch"

import torch
from torch.utils.data import random_split

# Manual training components (for advanced section)
from geomfum.convert import P2pFromFmConverter

# Datasets
from geomfum.dataset.torch import MeshDataset, PairsDataset
from geomfum.descriptor.learned import FeatureExtractor
from geomfum.descriptor.spectral import WaveKernelSignature

# Experiment framework
from geomfum.experiment import (
    Experiment,
    ExperimentConfig,
    ExperimentSuite,
    MatcherPresets,
    ModelPresets,
    TrainingPresets,
    quick_train,
)
from geomfum.forward_functional_map import ForwardFunctionalMap

# Presets (RECOMMENDED)
from geomfum.learning import (
    DeepFunctionalMapTrainer,
    LossManager,
)
from geomfum.learning.losses import (
    BijectivityLoss,
    GeodesicError,
    LaplacianCommutativityLoss,
    OrthonormalityLoss,
)
from geomfum.learning.models import FMNet
from geomfum.matcher import FunctionalMapMatcher

## Step 1: Prepare Training and Test Data

We'll use separate train and test sets to demonstrate the full workflow.

In [2]:
# Training data (no distances needed - faster)
TRAIN_PATH = "../../../datasets/faust/train_set/"
train_shapes = MeshDataset(
    dataset_dir=TRAIN_PATH,
    spectral=True,
    distances=False,  # Not needed for training
    correspondences=False,  # Using unsupervised losses
    device="cuda" if torch.cuda.is_available() else "cpu",
    k=30,
)

# Test data (with distances for evaluation)
TEST_PATH = "../../../datasets/faust/test_set/"
test_shapes = MeshDataset(
    dataset_dir=TEST_PATH,
    spectral=True,
    distances=True,  # Needed for geodesic error metric
    correspondences=True,  # Ground truth for evaluation
    device="cuda" if torch.cuda.is_available() else "cpu",
    k=30,
)

print(f"Training shapes: {len(train_shapes)}")
print(f"Test shapes: {len(test_shapes)}")

/home/ubuntu/giulio_vigano/geomfum_proj/venv/lib/python3.12/site-packages/gsops/pytorch/sparse.py:21: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  return _torch.sparse_csc_tensor(ccol_indices, row_indices, values, size=array.shape)


Training shapes: 80
Test shapes: 20


In [8]:
# Create shape pairs
train_pairs = PairsDataset(train_shapes, pair_mode="all")

# Split training pairs into train/val
train_size = int(0.8 * len(train_pairs))
val_size = len(train_pairs) - train_size
train_dataset, val_dataset = random_split(train_pairs, [train_size, val_size])

# Test pairs (small subset for quick evaluation)
test_pairs = PairsDataset(test_shapes, pairs_ratio=0.1)

print(f"Train pairs: {len(train_dataset)}")
print(f"Val pairs: {len(val_dataset)}")
print(f"Test pairs: {len(test_pairs)}")

Train pairs: 5056
Val pairs: 1264
Test pairs: 380


## Method 1: Quick Training with Presets (RECOMMENDED)

The fastest way to train! Just like `MatcherPresets.get("standard")` for classical methods, we have preset configurations for learning pipelines.

In [ ]:
# See available presets
print("Model Presets:")
for preset in ModelPresets.list_presets():
    config = ModelPresets.describe(preset)
    print(
        f"  {preset}: k={config['feature_extractor']['k_eig']}, channels={config['feature_extractor']['in_channels']}"
    )

print("\nTraining Presets:")
for preset in TrainingPresets.list_presets():
    config = TrainingPresets.describe(preset)
    print(f"  {preset}: {config['training']['epochs']} epochs")

Model Presets:
  fmnet_diffusion_large: k=300, channels=256
  fmnet_diffusion_small: k=128, channels=64
  fmnet_diffusion_standard: k=200, channels=128

Training Presets:
  supervised: 50 epochs
  supervised_plus_regularization: 50 epochs
  unsupervised_precise: 100 epochs
  unsupervised_quick: 5 epochs
  unsupervised_standard: 50 epochs


In [22]:
# One-liner training setup!
trainer = quick_train(
    preset="unsupervised_standard",
    train_set=train_dataset,
    val_set=val_dataset,
    model_preset="fmnet_diffusion_standard",
    checkpoint_path="checkpoints/fmnet_preset.pth",
)

print("Trainer ready! This single line configured:")
print("  - FMNet model with DiffusionNet features")
print("  - Unsupervised losses (Orthonormality, Bijectivity, Laplacian)")
print("  - Adam optimizer with lr=1e-3")
print("  - 50 epochs with GeodesicError monitoring")
print("\nNow call trainer.train() to start training!")

Trainer ready! This single line configured:
  - FMNet model with DiffusionNet features
  - Unsupervised losses (Orthonormality, Bijectivity, Laplacian)
  - Adam optimizer with lr=1e-3
  - 50 epochs with GeodesicError monitoring

Now call trainer.train() to start training!


In [28]:
# Override specific parameters
trainer_custom = quick_train(
    preset="unsupervised_quick",  # Faster preset for quick testing
    train_set=train_dataset,
    val_set=val_dataset,
    model_preset="fmnet_diffusion_small",  # Smaller model
    epochs=20,  # Override: more epochs than 'quick' preset
    lr=5e-4,  # Override: lower learning rate
    checkpoint_path="checkpoints/fmnet_custom.pth",
)

print("Custom trainer configured with overrides!")

Custom trainer configured with overrides!


In [30]:
trainer.train()

INFO:root:Epoch [1/50] - Training
Epoch 1/50 (Train):   0%|          | 0/5056 [00:05<?, ?batch/s]


RuntimeError: Only Tensors created explicitly by the user (graph leaves) support the deepcopy protocol at the moment.  If you were attempting to deepcopy a module, this may be because of a torch.nn.utils.weight_norm usage, see https://github.com/pytorch/pytorch/pull/103001

## Method 2: Step-by-Step with Presets

For more control, build the model and trainer separately using presets.

In [24]:
# Step 1: Build model from preset
model = ModelPresets.build(
    "fmnet_diffusion_standard", device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Model built from preset!")
print(f"  Device: {next(model.parameters()).device}")

Model built from preset!
  Device: cuda:0


In [25]:
# Step 2: Create trainer from preset
trainer_stepwise = TrainingPresets.create_trainer(
    preset="unsupervised_standard",
    model=model,
    train_set=train_dataset,
    val_set=val_dataset,
    checkpoint_path="checkpoints/fmnet_stepwise.pth",
)

print("Trainer configured!")
print(f"  Training losses: {len(trainer_stepwise.train_loss_manager.losses)}")
print(f"  Validation losses: {len(trainer_stepwise.val_loss_manager.losses)}")
print(f"  Epochs: {trainer_stepwise.epochs}")
print(f"  Learning rate: {trainer_stepwise.optimizer.param_groups[0]['lr']}")

Trainer configured!
  Training losses: 3
  Validation losses: 1
  Epochs: 50
  Learning rate: 0.001


## Method 3: Comparing Different Training Presets

Just like comparing matcher presets, we can train models with different configurations.

In [26]:
# Train multiple models with different presets
training_configs = {
    "quick": "unsupervised_quick",
    "standard": "unsupervised_standard",
    "supervised": "supervised",
}

# This would train all models (commented out for demo)
# trained_models = {}
# for name, preset in training_configs.items():
#     trainer = quick_train(
#         preset=preset,
#         train_set=train_dataset,
#         val_set=val_dataset,
#         checkpoint_path=f"checkpoints/fmnet_{name}.pth",
#     )
#     trainer.train()
#     trained_models[name] = load_trained_model(
#         f"checkpoints/fmnet_{name}.pth",
#         trainer.model
#     )

print("Multiple models can be trained with different presets and compared!")

Multiple models can be trained with different presets and compared!


## Method 4: Manual Training Setup (Advanced)

For full control over every component, you can build models and trainers manually.
This is useful when implementing custom architectures or loss functions.

In [ ]:
# Build model components
device = "cuda" if torch.cuda.is_available() else "cpu"

# Feature extractor (DiffusionNet)
feature_extractor = FeatureExtractor.from_registry(
    which="diffusionnet",
    device=device,
    k=200,
    in_channels=128,
    descriptor=WaveKernelSignature(n_domain=128),
)

# Functional map module
fmap_module = ForwardFunctionalMap(weight=1e3, reduce_dim=1, bidirectional=True)

# Complete model
model = FMNet(
    feature_extractor=feature_extractor,
    fmap_module=fmap_module,
    converter=P2pFromFmConverter(),
)

print(f"Model built on device: {device}")

TypeError: DiffusionnetFeatureExtractor.__init__() got an unexpected keyword argument 'k_eig'

In [ ]:
# Define training losses (unsupervised)
train_losses = [
    OrthonormalityLoss(weight=1.0),
    BijectivityLoss(weight=1.0),
    LaplacianCommutativityLoss(weight=1e-3),
]
train_loss_manager = LossManager(train_losses)

# Define validation losses
val_losses = [
    GeodesicError(),  # Main metric for evaluation
]
val_loss_manager = LossManager(val_losses)

print("Loss managers configured")

In [ ]:
# Setup optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# Create trainer
trainer = DeepFunctionalMapTrainer(
    model=model,
    train_set=train_dataset,
    val_set=val_dataset,
    train_loss_manager=train_loss_manager,
    val_loss_manager=val_loss_manager,
    optimizer=optimizer,
    device=device,
    epochs=10,  # Use more epochs for real training
    checkpoint_path="checkpoints/fmnet_best.pth",
    monitor_metric="GeodesicError",
    mode="min",
)

print("Trainer configured")

In [ ]:
# Train the model
# trainer.train()

# Uncomment above to actually train
# For this demo, we assume a checkpoint exists
print("Training complete! (or use existing checkpoint)")

## Evaluating Trained Models

After training (with any method above), load and evaluate your model.
Trained models integrate seamlessly with the experiment framework!

In [ ]:
# Load the trained model
# If you used presets, you can rebuild the model from preset
model_for_eval = ModelPresets.build("fmnet_diffusion_standard")

trained_model = load_trained_model(
    checkpoint_path="checkpoints/fmnet_preset.pth",
    model=model_for_eval,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("Trained model loaded and ready for evaluation!")

In [ ]:
# Run experiment on trained model
experiment = Experiment(
    method=trained_model,
    dataset=test_pairs,
    config=ExperimentConfig(name="FMNet_Trained", progress_bar=True),
)

result = experiment.run()

# View results
print("\nTrained Model Results:")
for metric, value in result.metrics.items():
    if not metric.endswith("_std"):
        std = result.metrics.get(f"{metric}_std", 0)
        print(f"  {metric}: {value:.4f} ± {std:.4f}")

## Comparing Trained Models with Classical Matchers

The key advantage of the preset system: both learning-based and classical methods use the same preset pattern!

Compare models trained with different presets against classical matcher presets.

In [ ]:
# Build a comprehensive comparison
# Both learning-based models and classical matchers use presets!
methods = {
    # Trained models (would load actual checkpoints)
    "FMNet (unsupervised)": trained_model,
    # Classical matchers with presets
    "FunctionalMap (quick)": FunctionalMapMatcher(config=MatcherPresets.get("quick")),
    "FunctionalMap (standard)": FunctionalMapMatcher(
        config=MatcherPresets.get("standard")
    ),
    "FunctionalMap (precise)": FunctionalMapMatcher(
        config=MatcherPresets.get("precise")
    ),
}

# If you had trained multiple models with different presets, you could add:
# model_quick = ModelPresets.build("fmnet_diffusion_small")
# methods["FMNet (quick)"] = load_trained_model("checkpoints/fmnet_quick.pth", model_quick)
#
# model_large = ModelPresets.build("fmnet_diffusion_large")
# methods["FMNet (precise)"] = load_trained_model("checkpoints/fmnet_precise.pth", model_large)

# Run comparison
suite = ExperimentSuite(methods, test_pairs)
all_results = suite.run()

In [ ]:
# Print comparison table
suite.print_comparison(metrics=["geodesic_error", "coverage", "dirichlet_energy"])

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt
import numpy as np

method_names = list(all_results.keys())
geo_errors = [all_results[m].metrics["geodesic_error"] for m in method_names]
geo_stds = [all_results[m].metrics["geodesic_error_std"] for m in method_names]

fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(method_names))

bars = ax.bar(x_pos, geo_errors, yerr=geo_stds, capsize=5, alpha=0.7)
ax.set_xlabel("Method")
ax.set_ylabel("Geodesic Error")
ax.set_title("Trained Model vs Classical Matchers")
ax.set_xticks(x_pos)
ax.set_xticklabels(method_names, rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

# Highlight trained model
bars[0].set_color("red")
bars[0].set_alpha(0.9)

plt.tight_layout()
plt.show()

## Alternative: Direct Model Usage (Without Wrapper)

If your model already follows the interface, you can use it directly.

In [ ]:
# If your model has .eval() and returns CorrespondenceResult,
# it works directly without wrapping!

# Load checkpoint manually
checkpoint = torch.load("checkpoints/fmnet_best.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Use directly in experiment
# The Experiment class auto-detects it's a model
experiment = Experiment(model, test_pairs)
# result = experiment.run()  # Works!

## Best Practices

### 1. Use Presets (RECOMMENDED)

**Quick training (one-liner):**
```python
from geomfum.learning import quick_train

trainer = quick_train(
    preset="unsupervised_standard",
    train_set=train_pairs,
    val_set=val_pairs,
)
trainer.train()
```

**Step-by-step with presets:**
```python
from geomfum.learning import ModelPresets, TrainingPresets

model = ModelPresets.build("fmnet_diffusion_standard")
trainer = TrainingPresets.create_trainer(
    preset="unsupervised_standard",
    model=model,
    train_set=train_pairs,
    val_set=val_pairs,
)
trainer.train()
```

**Available Presets:**
- **Model**: fmnet_diffusion_small, fmnet_diffusion_standard, fmnet_diffusion_large
- **Training**: unsupervised_quick, unsupervised_standard, unsupervised_precise, supervised, supervised_plus_regularization

### 2. Training Data vs Evaluation Data
- **Training**: No need for distances/correspondences (use unsupervised losses)
- **Evaluation**: Include distances and correspondences for proper metrics

### 3. Override Parameters
```python
trainer = quick_train(
    preset="unsupervised_standard",
    train_set=train_pairs,
    val_set=val_pairs,
    epochs=100,  # Override
    lr=5e-4,     # Override
)
```

### 4. Comparing Models and Matchers
Both use the same preset pattern!
```python
methods = {
    "FMNet": load_trained_model("checkpoint.pth", ModelPresets.build("fmnet_diffusion_standard")),
    "Classical": FunctionalMapMatcher(config=MatcherPresets.get("standard")),
}
ExperimentSuite(methods, dataset).run()
```

### 5. Model Interface Requirements
Your model must:
- Have a `__call__(shape_a, shape_b, bidirectional=False)` method
- Return a `CorrespondenceResult` object
- Have an `eval()` method (standard for PyTorch models)

### 6. Saving Results
```python
# Save experiment results
result.save("results/fmnet_evaluation.json")

# Save all methods
suite.save_all("results/comparison/")
```

## Summary

**The Preset Pattern Throughout GeomFUM:**

GeomFUM uses a consistent preset system across all components:

| Component | Preset Class | Example |
|-----------|-------------|---------|
| **Classical Matchers** | `MatcherPresets` | `MatcherPresets.get("standard")` |
| **Model Architecture** | `ModelPresets` | `ModelPresets.build("fmnet_diffusion_standard")` |
| **Training Pipeline** | `TrainingPresets` | `TrainingPresets.create_trainer("unsupervised_standard", ...)` |
| **Quick Training** | `quick_train()` | `quick_train("unsupervised_standard", ...)` |

**Training Workflow:**

1. **Quick Start** (one-liner):
   ```python
   trainer = quick_train("unsupervised_standard", train_pairs, val_pairs)
   trainer.train()
   ```

2. **Step-by-step** (more control):
   ```python
   model = ModelPresets.build("fmnet_diffusion_standard")
   trainer = TrainingPresets.create_trainer("unsupervised_standard", model, train_pairs, val_pairs)
   trainer.train()
   ```

3. **Manual** (full control):
   - Build model components manually
   - Configure losses and optimizer
   - Create trainer

**Evaluation Workflow:**

1. Load trained model: `load_trained_model(checkpoint_path, model)`
2. Run experiments: `Experiment(model, dataset).run()`
3. Compare methods: `ExperimentSuite(methods, dataset).run()`

**Key Advantage:**

Unified preset system means you can easily:
- Switch between learning-based and classical methods
- Compare different configurations systematically
- Share reproducible configurations
- Quickly prototype new ideas

## Next Steps

- **[21_experiment.ipynb](./21_experiment.ipynb)** - Basic experiment framework
- **[22_configuration_presets.ipynb](./22_configuration_presets.ipynb)** - Matcher preset details
- **[23_systematic_experiments.ipynb](./23_systematic_experiments.ipynb)** - Grid search for hyperparameters
- **Deep Functional Maps Demo** - See `docs/notebooks/demos/Deep-Functional-Maps-Demo.ipynb` for full training example